# nb07 — PCA / UMAP / tSNE of the approaches selected by the rule: unharmonized vs harmonized

**Plan:** `article_1_revision_implementation_plan_261029.md`, WP4 step 4.2b (gate G2b) and WP9.
**Request (Daniil, G2b 2026-11-03, `gates_review_261102.md` §4.1):** draw **every** approach
selected by the frozen rule on PCA / UMAP / tSNE, coloured by `Diagnosis_cell_type_unified` and by
`RNA_BATCH`, **next to the same plots of the unharmonized data of the same strategy and
imputation**. When the approach uses post-removal, the batches it removes are drawn faint. Daniil
decides the vetoes from these plots: do batches mix within the same biology?

**What is drawn.** For each selected approach one *pair* of columns:

| Column | Matrix | Faint points |
|---|---|---|
| left | unharmonized (`01_raw`) of the same strategy and imputation, without post-removal | the batches the approach's post-removal drops (none if post-removal is off) |
| right | the approach's harmonized matrix **before** post-removal | the same batches |

Post-removal only deletes rows (whole outlier batches) from the harmonized matrix
(`harmonization-scripts/run_norm_job.py`, `identify_outlier_batches`), so the matrix before removal
is the approach's matrix plus those rows. Decision (Daniil, 2026-11-03): embed that full matrix and
fade the removed batches, so their position is visible; they also take part in the layout.

**Scenarios (pools of `r04_apply_rule.py`):** the four tree leaves, the information-only
"Microarrays only" pool and the information-only "No batch removal (S0 only)" pool (G2b request,
both post-removal settings). Rows: batch, diagnosis, and platform for the three pools that mix
platforms (rule step 4c: leaf assignment of the multi-platform front).

**Method (as the submitted visual inspection,
`git show 58c5566:harmonization-metrics/harmonization_metrics_visual_inspection.ipynb`, cell 18):**
genes with any missing value dropped → `StandardScaler` → PCA (50 components); UMAP on the 50 PCs
(`n_neighbors = 30`, `min_dist = 0.3`, `random_state = 42`); tSNE on the 50 PCs
(`perplexity = 30`, `random_state = 42`). Points `s = 3`, `alpha = 0.5` (faint: `alpha = 0.06`),
rasterized.

**Review rounds.** Round 1 (2026-11-03) drew the 53 version-1 front members; Daniil's decisions
(2026-11-04) are in `output/wp4/visual_check.tsv` (vetoed rows also in `vetoes.tsv`). r04 then
recomputes each front without the vetoed approaches; approaches that enter only then are drawn in
round 2 (`REVIEW_ROUND = 2`, the default now).

**Blindness.** Reads `output/wp4/selection_by_rule.tsv` (round 1) or
`output/wp4/selection_after_visual_check.tsv` (round 2), written by r04 after the freeze (gate G2);
computes no front itself.

**Inputs:** `output/wp4/selection_by_rule.tsv`; S3 `FL_batch_correction/exp/…` matrices and
`prepared/{strat}__{imp}__ann.tsv.gz`; the 7,174-sample annotation (platform).
**Outputs:** `figures_svg/g2b_review/<scenario>_{pca,umap,tsne}.pdf` (one page per 3 pairs),
`output/wp9/nb07_selected_approaches.tsv`, `output/wp9/nb07_removed_batches.tsv`; embeddings cached
in `cache/nb07/` (git-ignored).

**Run (≈ 3–5 h when nothing is cached; tSNE dominates):**
`jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=collagen_3_11 nb07_selected_approaches_embeddings.ipynb`

## §0 Configuration

### §0a Imports

In [ ]:
# Imports. revision_lib is used only for plain-word names, S3 reads and paths (plan WP9).
import hashlib
import re
import textwrap

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from joblib import Parallel, delayed
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import revision_lib as rl

### §0b Paths and switches

| Switch | Default | Meaning |
|---|---|---|
| `REVIEW_ROUND` | `"final"` | `"final"` = the 22 accepted approaches of the final selection (→ `review_extras/final_selection_embeddings/`, table `nb07_figure_files_final.tsv`; all embeddings cached, no new compute); `1` = every version-1 front member (`selection_by_rule.tsv`, → `figures_svg/g2b_review/`); `2` = only the approaches that entered the front after the round-1 vetoes (`selection_after_visual_check.tsv`, `visual_check == "not reviewed"`, → `figures_svg/g2b_review_round2/`, tables `…_round2.tsv`) |
| `SCENARIOS_TO_DRAW` | `None` | `None` = all six pools of `selection_by_rule.tsv` (Daniil, G2b: tree leaves + both information-only pools); or a list of scenario names |
| `TOP_PER_SCENARIO` | `None` | `None` = every front member; a number keeps the first n in rule order |
| `PAIRS_PER_PAGE` | `3` | approaches per PDF page (each one is an unharmonized + harmonized pair → 6 columns) |
| `PLATFORM_ROW_SCENARIOS` | the three pools that mix platforms | third row coloured by platform |
| `FAINT_ALPHA`, `POINT_ALPHA`, `POINT_SIZE` | 0.06, 0.5, 3 | faint = batches removed by the approach's post-removal |
| `N_JOBS`, `TSNE_THREADS` | 4, 2 | parallel embeddings on JupyterHub (8 CPUs, 61 GB) |
| `N_PCS`, `UMAP_NEIGHBORS`, `UMAP_MIN_DIST`, `TSNE_PERPLEXITY`, `SEED` | 50, 30, 0.3, 30, 42 | the submitted embedding settings |

In [ ]:
# Paths and switches.
# Review round (gate G2b). 1 = every version-1 front member (selection_by_rule.tsv);
# 2 = the approaches that entered the front after the round-1 vetoes and were never shown
# (selection_after_visual_check.tsv, visual_check == "not reviewed").
# "final" = the 22 accepted approaches of the final selection (Daniil 2026-11-08: "where are the
# PCA / UMAP / tSNE plots of the best approaches only?"); cache only, no new embedding.
REVIEW_ROUND = "final"
SELECTION_PATH = rl.OUTPUT_DIR / (
    "wp4/selection_by_rule.tsv" if REVIEW_ROUND == 1 else "wp4/selection_after_visual_check.tsv"
)
ROUND_SUFFIX = {1: "", 2: "_round2", "final": "_final"}[REVIEW_ROUND]
FIGURES_DIR = (rl.REVISION_DIR / "review_extras/final_selection_embeddings" if REVIEW_ROUND == "final"
               else rl.REVISION_DIR / f"figures_svg/g2b_review{ROUND_SUFFIX}")
TABLES_DIR = rl.OUTPUT_DIR / "wp9"
CACHE_DIR = rl.REVISION_DIR / "cache/nb07"
for folder in (FIGURES_DIR, TABLES_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SCENARIOS_TO_DRAW = None  # None = all six pools
TOP_PER_SCENARIO = None  # None = every front member
PAIRS_PER_PAGE = 3
PLATFORM_ROW_SCENARIOS = {
    "Multi-platform (RNA-seq + microarrays)",
    "No batch removal (S0 only)",
    "Microarrays only (not a tree leaf)",
}
FAINT_ALPHA = 0.06
POINT_ALPHA = 0.5
POINT_SIZE = 3
N_JOBS = 4
TSNE_THREADS = 2
N_PCS = 50
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST = 0.3
TSNE_PERPLEXITY = 30
SEED = 42
GLOBAL_FONT_SIZE = 12
TITLE_WRAP = 30  # characters per title line
APPROACH_KEYS = ["strat", "imp", "method", "post_rm"]

### §0c Palettes and plain-word names

Batch and diagnosis palettes are copied from `nb01_confounding.ipynb` (the Article 1 palettes:
Temperature-Split, tumours warm, normal B cells cold). A level without a colour gets a
deterministic MD5 colour, and every such assignment is printed. Labels come from `revision_lib`
(plan §1.1).

In [ ]:
# Article 1 palettes (copied, not imported)
import matplotlib.colors as mcolors

lymphoma_ontogeny_palette = {
    "Bone_marrow_CD19+": "blue",
    "Immature": "#08519c",
    "Naive": "#4292c6",
    "GC": "#006d2c",
    "Centroblast": "#41ab5d",
    "Centrocyte": "#a1d99b",
    "B_cells": "lawngreen",
    "MZ": "olive",
    "Memory": "#00ced1",
    "Plasmablast": "#5f9ea0",
    "Plasma": "grey",
    "Follicular_Lymphoma": "#6a0dad",
    "Follicular_Lymphoma_GCB": "#9400d3",
    "Diffuse_Large_B_Cell_Lymphoma_ABC": "#ff4500",
    "Diffuse_Large_B_Cell_Lymphoma_GCB": "#d62728",
    "Diffuse_Large_B_Cell_Lymphoma": "#8b0000",
    "High_Grade_B_Cell_Lymphoma": "#ff00ff",
    "Burkitt_Lymphoma": "orange",
}

rna_batch_palette = {
    "RNASeq_FF_Total": "#1f77b4",
    "RNASeq_FF_PolyA": "#17becf",
    "RNASeq_FFPE_Exome_capture": "#9467bd",
    "RNASeq_FF_rRNADepletion": "#000080",
    "RNASeq_FFPE_PolyA": "#8c564b",
    "RNASeq_FF_Unknown": "#bcbd22",
    "RNASeq_FF_Exome_capture": "#2ca02c",
    "GPL96+97_FF_Unknown": "#d62728",
    "GPL570_FF_Unknown": "#ff7f0e",
    "GPL570_Unknown_Unknown": "#ffbb78",
    "GPL570_FFPE_Unknown": "#dbdb8d",
    "GPL96_FF_Unknown": "#f7b6d2",
    "GPL96_FFPE_Unknown": "#e377c2",
    "GPL6244_FF_Unknown": "#e9967a",
    "GPL6244_FFPE_Unknown": "#fa8072",
    "GPL13938_FFPE_Unknown": "#800000",
    "GPL1708_FF_Unknown": "#7f7f7f",
    "GPL14951_FFPE_Unknown": "#c49c94",
    "GPL17077_FF_Unknown": "#c5b0d5",
    "GPL13158_FF_Unknown": "#ad494a",
    "GPL17586_FF_Unknown": "#008b8b",
    "GPL17047_FF_Unknown": "#4682b4",
    "GPL887_FFPE_Unknown": "#b0c4de",
    "GPL10739_FF_Unknown": "#9edae5",
    "GPL8432_FFPE_Unknown": "#006400",
    "GPL20188_FF_Unknown": "#556b2f",
    "GPL23541_FF_Unknown": "#8fbc8f",
    "GPL16686_FF_Unknown": "#333333",
    "GPL26356_FF_Unknown": "#a0522d",
}

strat_pal = {
    "S0_no_removal": (0.6313725490196078, 0.788235294117647, 0.9568627450980393),
    "A_confirmed_bad": (1.0, 0.7058, 0.5098),
    "B_extended_bad": (0.5529, 0.8980, 0.6313),
    "C_rnaseq_only": (1.0, 0.6235, 0.6078),
    "D_malignant_only": (0.8156, 0.7333, 1.0),
    "E1_iterative_r1": (0.8705, 0.7333, 0.6078),
    "E2_iterative_r2": (0.9803, 0.6901, 0.8941),
    "E3_iterative_r3": (0.8117, 0.8117, 0.8117),
    "F_microarray_only": (1.0, 0.9960, 0.6392),
    "G_affymetrix_only": (0.7254, 0.9490, 0.9411),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only": "#7ecbc4",
    "K_ffpe_only": "#f2c27f",
}

IMPUTATION_NAMES = {"strict": "no imputation", "knn": "KNN", "softimpute": "softImpute"}
MISSING_COLOR = "#AAAAAA"

### §0d Helpers and plot settings

In [ ]:
# Plot settings (project CLAUDE.md).
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = GLOBAL_FONT_SIZE
sns.set_style("ticks")


def level_color(level: str, palette: dict, prefix_match: bool) -> str:
    # exact match for batches, prefix match for diagnoses; otherwise a deterministic MD5 colour
    if pd.isna(level):
        return MISSING_COLOR
    if level in palette:
        return palette[level]
    if prefix_match:
        for key, colour in palette.items():
            if str(level).startswith(key):
                return colour
    colour = "#" + hashlib.md5(str(level).encode()).hexdigest()[:6]
    palette[level] = colour  # print each assignment once
    print(f"palette: {level} → {colour} (MD5)")
    return colour


def approach_title(strat: str, imp: str, method: str, post_rm: bool) -> str:
    # plain-word panel title: method, then strategy · imputation (· post-removal), wrapped
    first = rl.method_display_name(method) if method != "01_raw" else "Unharmonized"
    rest = [rl.STRATEGY_DISPLAY_NAMES.get(strat, strat), IMPUTATION_NAMES.get(imp, imp)]
    if post_rm:
        rest.append("with post-removal")
    return "\n".join([first, textwrap.fill(" · ".join(rest), TITLE_WRAP)])


def file_stem(scenario: str) -> str:
    # scenario name → file name
    return re.sub(r"[^a-z0-9]+", "_", scenario.lower()).strip("_")

## §1 Which approaches to draw

The front members of every pool in `selection_by_rule.tsv`, in rule order (`front_rank`). Expected
(r04 run of 2026-11-03): FF only 8, FFPE only 5, RNA-seq only 8, multi-platform 12, microarrays
only 13, no batch removal 7 → 53 approaches.

In [ ]:
# Read the rule's selection (written by r04 after the freeze).
if not SELECTION_PATH.exists():
    raise SystemExit(f"{SELECTION_PATH} missing: run `python r04_apply_rule.py --review` first.")
selection = pd.read_csv(SELECTION_PATH, sep="\t").sort_values(["scenario", "front_rank"])
selection["post_rm"] = selection["post_rm"].astype(str).str.lower().isin(["true", "1"])
if REVIEW_ROUND == 2:
    selection = selection[selection["visual_check"] == "not reviewed"]
if REVIEW_ROUND == "final":
    selection = selection[selection["visual_check"] == "accepted"]
    assert len(selection) == 22, "final selection: 22 accepted approaches (G2b closed 2026-11-07)"
if SCENARIOS_TO_DRAW is not None:
    selection = selection[selection["scenario"].isin(SCENARIOS_TO_DRAW)]
if TOP_PER_SCENARIO is not None:
    selection = selection.groupby("scenario").head(TOP_PER_SCENARIO)
selection[["scenario", "front_rank", *APPROACH_KEYS, "rule_md5"]].to_csv(
    TABLES_DIR / f"nb07_selected_approaches{ROUND_SUFFIX}.tsv", sep="\t", index=False
)
print(selection.groupby("scenario", sort=False).size().to_string())
print(f"{len(selection)} approaches; rule MD5 {selection['rule_md5'].unique()}")
if REVIEW_ROUND == 2:
    assert len(selection) > 0, "nothing to draw"  # empty since the visual check closed (2026-11-07)

## §2 Embeddings (cached, parallel)

One embedding per matrix, cached as `cache/nb07/{strat}__{imp}__{method}__post0.pkl`:

- every approach's harmonized matrix **without** post-removal (`post0`; a post-removal approach
  uses the same matrix, see the title cell);
- every unharmonized matrix (`01_raw`, `post0`) of a strategy × imputation that appears.

Each: matrix + annotation from S3 → genes with any NA dropped → standardize → PCA (50) → UMAP and
tSNE on the PCs. `N_JOBS` matrices run in parallel; a re-run only re-draws.

In [ ]:
# Compute (or read) the PCA / UMAP / tSNE coordinates of one matrix (always post0).
def embedding_coordinates(strat: str, imp: str, method: str) -> pd.DataFrame:
    # samples × (PC1, PC2, UMAP1, UMAP2, tSNE1, tSNE2, batch, diagnosis); cached as a pickle
    cache_path = CACHE_DIR / f"{strat}__{imp}__{method}__post0.pkl"
    if cache_path.exists():
        return pd.read_pickle(cache_path)
    s3 = rl.s3_client()  # one client per worker process
    expression = rl.s3_get_tsv_gz(s3, rl.main_benchmark_matrix_key(strat, imp, method, False))
    annotation = rl.s3_get_tsv_gz(s3, f"{rl.S3_PREFIX}/prepared/{strat}__{imp}__ann.tsv.gz")
    shared = expression.index.intersection(annotation.index)
    expression = expression.loc[shared].dropna(axis=1)  # genes with any NA dropped
    scaled = StandardScaler().fit_transform(expression.to_numpy(dtype=float))
    pca = PCA(n_components=min(N_PCS, min(scaled.shape) - 1), random_state=SEED)
    pcs = pca.fit_transform(scaled)
    umap_xy = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                        n_components=2, random_state=SEED).fit_transform(pcs)
    tsne_xy = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY, random_state=SEED,
                   n_jobs=TSNE_THREADS).fit_transform(pcs)
    coordinates = pd.DataFrame(
        {"PC1": pcs[:, 0], "PC2": pcs[:, 1], "UMAP1": umap_xy[:, 0], "UMAP2": umap_xy[:, 1],
         "tSNE1": tsne_xy[:, 0], "tSNE2": tsne_xy[:, 1],
         "batch": annotation.loc[shared, rl.BATCH_COL].values,
         "diagnosis": annotation.loc[shared, rl.BIO_COL].values},
        index=shared,
    )
    coordinates.attrs["pc_variance"] = pca.explained_variance_ratio_[:2].tolist()
    coordinates.attrs["n_genes"] = int(expression.shape[1])
    coordinates.to_pickle(cache_path)
    return coordinates

In [ ]:
# Every matrix to embed: the approaches (post0) and their unharmonized counterparts.
matrices = sorted(
    {(r.strat, r.imp, r.method) for r in selection.itertuples()}
    | {(r.strat, r.imp, "01_raw") for r in selection.itertuples()}
)
missing = [m for m in matrices if not (CACHE_DIR / f"{m[0]}__{m[1]}__{m[2]}__post0.pkl").exists()]
print(f"{len(matrices)} matrices, {len(matrices) - len(missing)} cached, {len(missing)} to compute")
Parallel(n_jobs=N_JOBS, verbose=5)(delayed(embedding_coordinates)(*m) for m in missing)
coordinates_by_matrix = {m: embedding_coordinates(*m) for m in matrices}
for matrix, frame in coordinates_by_matrix.items():
    print(f"{matrix}: {len(frame)} samples, {frame.attrs.get('n_genes')} genes")

## §3 Batches removed by post-removal

For each approach with post-removal, the removed samples = rows of the matrix without
post-removal that are missing from the matrix with post-removal. The kept sample list is read from
an old nb07 cache (`…__post1.pkl`, same row set) when it exists, otherwise from the S3 matrix
(first column only), and cached as `…__post1_kept.txt`. Expect whole batches.

In [ ]:
# Kept samples of a post-removal approach (cached).
def kept_samples_after_post_removal(strat: str, imp: str, method: str) -> pd.Index:
    # sample IDs present in the post1 matrix
    kept_path = CACHE_DIR / f"{strat}__{imp}__{method}__post1_kept.txt"
    old_cache = CACHE_DIR / f"{strat}__{imp}__{method}__post1.pkl"
    if kept_path.exists():
        return pd.Index(kept_path.read_text().split())
    if old_cache.exists():
        kept = pd.read_pickle(old_cache).index
    else:
        key = rl.main_benchmark_matrix_key(strat, imp, method, True)
        kept = rl.s3_get_tsv_gz(rl.s3_client(), key).index  # whole matrix; read once
    kept_path.write_text("\n".join(map(str, kept)))
    return pd.Index(kept)


removed_rows = []
removed_by_approach = {}
for r in selection.drop_duplicates(APPROACH_KEYS).itertuples():
    approach = (r.strat, r.imp, r.method, r.post_rm)
    frame = coordinates_by_matrix[(r.strat, r.imp, r.method)]
    if not r.post_rm:
        removed_by_approach[approach] = pd.Index([])
        continue
    kept = kept_samples_after_post_removal(r.strat, r.imp, r.method)
    removed = frame.index.difference(kept)
    removed_by_approach[approach] = removed
    batches = frame.loc[removed, "batch"].value_counts()
    whole = all((frame["batch"] == b).sum() == n for b, n in batches.items())
    removed_rows.append({"strat": r.strat, "imp": r.imp, "method": r.method,
                         "n_samples_before": len(frame), "n_removed": len(removed),
                         "n_removed_batches": len(batches), "whole_batches": whole,
                         "removed_batches": "; ".join(f"{b} ({n})" for b, n in batches.items())})
removed_table = pd.DataFrame(removed_rows)
removed_table.to_csv(TABLES_DIR / f"nb07_removed_batches{ROUND_SUFFIX}.tsv", sep="\t", index=False)
print(removed_table[["strat", "imp", "method", "n_removed", "n_removed_batches",
                     "whole_batches"]].to_string(index=False))
assert removed_table.empty or removed_table["whole_batches"].all(), "post-removal removed part of a batch"

## §4 Figures: unharmonized | harmonized pairs

One PDF per scenario and embedding (`<scenario>_{pca,umap,tsne}.pdf`), `PAIRS_PER_PAGE` approaches
per page. Rows: batch, diagnosis (+ platform). In each pair the left column is the unharmonized
data of the same strategy and imputation, the right column the approach; batches removed by the
approach's post-removal are faint in both. Points drawn in a fixed shuffled order so that no batch
is always on top.

In [ ]:
# Platform of every sample (GEO platform ID or RNA-seq), from the 7,174-sample annotation.
sample_platform = rl.load_s0_annotation()["PLATFORM_RNA"].replace({"RNASeq": "RNA-seq"})
platform_levels = sorted(sample_platform.dropna().unique(), key=lambda v: (v != "RNA-seq", v))
platform_palette = {
    level: mcolors.to_hex(plt.cm.tab20(i % 20))
    for i, level in enumerate([v for v in platform_levels if v != "RNA-seq"], start=1)
}
platform_palette["RNA-seq"] = "#1f77b4"
EMBEDDINGS = {"pca": ("PC1", "PC2"), "umap": ("UMAP1", "UMAP2"), "tsne": ("tSNE1", "tSNE2")}
AXIS_NAMES = {"pca": "PC", "umap": "UMAP", "tsne": "tSNE"}
ROW_SPECS = {"batch": (rna_batch_palette, False), "diagnosis": (lymphoma_ontogeny_palette, True),
             "platform": (platform_palette, False)}
print(f"{len(platform_levels)} platforms")

In [ ]:
# Draw one panel: kept samples at POINT_ALPHA, removed ones at FAINT_ALPHA underneath.
def batch_display_name(batch: str) -> str:
    """Plain batch name (Daniil 2026-11-09, same rule as nb01 / nb08): underscores → spaces;
    microarrays drop the last "Unknown" (library preparation is never known for a microarray),
    RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def draw_panel(ax, frame: pd.DataFrame, removed: pd.Index, column: str, kind: str) -> None:
    x_col, y_col = EMBEDDINGS[kind]
    palette, prefix = ROW_SPECS[column]
    order = np.random.default_rng(SEED).permutation(len(frame))  # fixed shuffled draw order
    frame = frame.iloc[order]
    is_removed = frame.index.isin(removed)
    for subset, alpha in ((frame[is_removed], FAINT_ALPHA), (frame[~is_removed], POINT_ALPHA)):
        if len(subset):
            ax.scatter(subset[x_col], subset[y_col], s=POINT_SIZE, alpha=alpha, linewidths=0,
                       c=[level_color(v, palette, prefix) for v in subset[column]], rasterized=True)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_box_aspect(1)  # every embedding scatter is a square (Daniil 2026-11-09)
    if kind == "pca":
        variance = frame.attrs.get("pc_variance", [np.nan, np.nan])
        ax.set_xlabel(f"PC1 ({100 * variance[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({100 * variance[1]:.1f}%)")
    else:
        ax.set_xlabel(f"{AXIS_NAMES[kind]} 1"); ax.set_ylabel(f"{AXIS_NAMES[kind]} 2")

In [ ]:
# One page: rows × (2 · pairs) columns, legends for the levels present.
def draw_page(scenario: str, approaches: list[tuple], kind: str) -> plt.Figure:
    rows = ["batch", "diagnosis"] + (["platform"] if scenario in PLATFORM_ROW_SCENARIOS else [])
    n_cols = 2 * len(approaches)
    fig, axes = plt.subplots(len(rows), n_cols, figsize=(3.4 * n_cols, 3.4 * len(rows) + 1.0),
                             squeeze=False)
    present = {row: set() for row in rows}
    any_removed = False
    for pair, approach in enumerate(approaches):
        strat, imp, method, post_rm = approach
        removed = removed_by_approach[approach]
        any_removed |= len(removed) > 0
        for side, matrix in enumerate([(strat, imp, "01_raw"), (strat, imp, method)]):
            frame = coordinates_by_matrix[matrix].copy()
            frame["platform"] = sample_platform.reindex(frame.index).values
            col = 2 * pair + side
            for row_index, row in enumerate(rows):
                draw_panel(axes[row_index, col], frame, removed, row, kind)
                present[row] |= set(frame[row].dropna())
            title = approach_title(strat, imp, "01_raw", False) if side == 0 else approach_title(*approach)
            if side == 1 and len(removed):
                title += f"\n(faint: {len(removed)} samples removed)"
            axes[0, col].set_title(title, fontsize=GLOBAL_FONT_SIZE - 1)
    for row_index, row in enumerate(rows):
        axes[row_index, 0].annotate(f"Coloured by {row}", xy=(-0.22, 0.5), xycoords="axes fraction",
                                    rotation=90, va="center", ha="center")
    batch_handles = [mpatches.Patch(color=level_color(b, rna_batch_palette, False), label=batch_display_name(b))
                     for b in sorted(present["batch"])]
    if any_removed:
        batch_handles.append(mpatches.Patch(color="grey", alpha=FAINT_ALPHA * 4,
                                            label="faint = removed by post-removal"))
    fig.legend(handles=batch_handles, title="Batch", loc="upper left", bbox_to_anchor=(1.0, 0.98),
               ncol=2, frameon=False, fontsize=GLOBAL_FONT_SIZE - 3)
    diagnosis_handles = [mpatches.Patch(color=level_color(d, lymphoma_ontogeny_palette, True),
                                        label=str(d).replace("_", " ")) for d in sorted(present["diagnosis"])]
    fig.legend(handles=diagnosis_handles, title="Diagnosis", loc="upper center",
               bbox_to_anchor=(0.5, 0.01), ncol=4, frameon=False, fontsize=GLOBAL_FONT_SIZE - 2)
    if "platform" in rows:
        platform_handles = [mpatches.Patch(color=platform_palette.get(p, MISSING_COLOR), label=p)
                            for p in sorted(present["platform"], key=lambda v: (v != "RNA-seq", v))]
        fig.legend(handles=platform_handles, title="Platform", loc="lower left",
                   bbox_to_anchor=(1.0, 0.02), ncol=2, frameon=False, fontsize=GLOBAL_FONT_SIZE - 3)
    fig.suptitle(scenario, fontsize=GLOBAL_FONT_SIZE + 2, y=1.0)
    fig.tight_layout()
    return fig

In [ ]:
# Write one multi-page PDF per scenario and embedding; show the first page inline.
figure_log = []
for scenario, picks in selection.groupby("scenario", sort=False):
    approaches = [tuple(r) for r in picks[APPROACH_KEYS].itertuples(index=False)]
    pages = [approaches[i:i + PAIRS_PER_PAGE] for i in range(0, len(approaches), PAIRS_PER_PAGE)]
    for kind in EMBEDDINGS:
        path = FIGURES_DIR / f"{file_stem(scenario)}_{kind}.pdf"
        with PdfPages(path) as pdf:
            for page_number, page in enumerate(pages, start=1):
                fig = draw_page(scenario, page, kind)
                pdf.savefig(fig, dpi=150, bbox_inches="tight")
                if page_number == 1 and kind == "umap":
                    plt.show()
                plt.close(fig)
        figure_log.append({"scenario": scenario, "embedding": kind, "pages": len(pages),
                           "approaches": len(approaches), "file": path.name,
                           "size_mb": round(path.stat().st_size / 1e6, 2)})
figure_log = pd.DataFrame(figure_log)
figure_log.to_csv(TABLES_DIR / f"nb07_figure_files{ROUND_SUFFIX}.tsv", sep="\t", index=False)
print(figure_log.to_string(index=False))
print(f"total {figure_log['size_mb'].sum():.1f} MB")

## §5 Where the PDFs of each round are

| Round | Content | Folder |
|---|---|---|
| 1 (2026-11-03) | all 53 version-1 front members, incl. SVA in FFPE only (front rank 1) and RNA-seq only (SVA KNN / softImpute, ranks 4–5) | `figures_svg/g2b_review/` |
| 2 (2026-11-07) | approaches that entered a front after the round-1 vetoes; empty once the visual check closed (why the saved notebook showed nothing) | `figures_svg/g2b_review_round2/` |
| final (2026-11-08) | the 22 accepted approaches of the final selection, unharmonized \| harmonized, coloured by batch, diagnosis (+ platform) | `review_extras/final_selection_embeddings/` |

Inline, the notebook only ever shows the first UMAP page of each PDF.

### §5b Open issues for Daniil (gate G2b, closed 2026-11-07)

0. **Round 2:** the decisions of round 1 are in `output/wp4/visual_check.tsv`; this round shows
   only the approaches that entered a front after those vetoes. Add their decisions there (and the
   vetoed ones to `vetoes.tsv`), then re-run r04; repeat until no front member is "not reviewed".
1. **Vetoes:** for each approach, do the batches mix within the same diagnosis (right column vs
   left column)? A broken approach gets a row in `output/wp4/vetoes.tsv`
   (`strat, imp, method, post_rm, reason`); then `python r04_apply_rule.py --review` and this
   notebook are re-run (the vetoed approach leaves the front, the next one may enter).
2. **Leaf assignment** of the multi-platform front members (platform row): RNA-seq + Illumina
   arrays only, or RNA-seq + various microarrays (rule step 4c).
3. **No batch removal (S0 only):** information only (asked after the freeze); which of its members
   to quote in the text, if any.
4. Which grids, if any, replace panels of the submitted Figures 4–5 / Supplementary Figures 11–17
   (decision D5; the figure count may not grow).